# Home-AI safety benchmarks: what the reported numbers measure, and what they miss

**Claim cluster:** `C031`, `C032`, `C033`, `C037`, `C038`, `C039`, `C040`, `C041`, `C042`, `C106`, `C107`, `C108`, `C119`  
**Source of truth:** `data/smarthome.duckdb` (read-only), via `src/viz_core.py`

In [ ]:
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, clear_output, display

# Find the repo root by walking up until schemas/stage2.sql appears, so the
# notebook runs from notebooks/, from the repo root, or from anywhere else.
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "schemas" / "stage2.sql").exists():
        ROOT = _candidate
        break
else:  # pragma: no cover - only reachable outside a checkout
    raise FileNotFoundError("schemas/stage2.sql not found above " + str(_here))

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import viz_core as viz  # noqa: E402

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# Renderers matter for two reasons. "browser" (plotly's default) opens a
# window when a figure is shown outside a live kernel, which breaks headless
# execution. The bare "notebook" renderer in plotly 7 emits only text/html,
# carrying no figure payload, so a notebook could run clean and still show
# nothing. "plotly_mimetype+notebook" emits the figure JSON for JupyterLab and
# an inline HTML fallback elsewhere -- which is also what the headless test
# asserts on.
pio.renderers.default = "plotly_mimetype+notebook"

# Every figure in these notebooks reads only from data/smarthome.duckdb.
CLAIM_IDS = ('C031', 'C032', 'C033', 'C037', 'C038', 'C039', 'C040', 'C041', 'C042', 'C106', 'C107', 'C108', 'C119')
assert viz.DEFAULT_DB_PATH.endswith("smarthome.duckdb"), viz.DEFAULT_DB_PATH

claims = viz.fetch_claims(list(CLAIM_IDS))
sources = viz.sources_for_claims(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
cluster_metrics = (
    viz.claim_metrics_lookup(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
)

# Corpus-wide row counts, read once so a notebook can state the size of what it
# is drawing from without confusing a whole-corpus count with the size of its
# own (possibly empty) claim cluster.
CORPUS = viz.connect_db()
try:
    corpus_counts = {
        table: CORPUS.execute(f"SELECT COUNT(1) FROM {table}").fetchone()[0]
        for table in ("stage2_claim", "metric", "entity", "source_mirror",
                      "predicate", "extraction_decision")
    }
finally:
    CORPUS.close()

def provenance(claim_id):
    """Collapsible evidence panel for one claim in this cluster."""
    panel = viz.render_provenance_panel(claim_id)
    return widgets.VBox([panel])

def show(*items):
    """Display data frames, HTML, or figures, one per argument."""
    for item in items:
        display(item)

def confidence_legend():
    """Show the colour coding applied to confidence across all figures."""
    display(HTML(
        "<b>Confidence colour coding:</b> "
        f"<span style='color:{viz.CONFIDENCE_COLORS['high']}'>high</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['medium']}'>medium</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['low']}'>low</span>"
    ))

## Provenance

Every result here comes from a research artefact the corpus registered as a source, not from testing of a commercial assistant: `C042` states that no commercial assistant publishes safety benchmark results. The provenance panels name the artefact behind each number, which is the only way to read the comparison chart honestly.

In [ ]:
for claim_id in CLAIM_IDS:
    display(HTML(f"<h4>{claim_id} &mdash; {claims.set_index('local_id').loc[claim_id, 'claim_text']}</h4>"))
    display(provenance(claim_id))
confidence_legend()

## Primary evidence

### Reported success rates, against a baseline that does nothing

`M079` records a baseline of 29.98% exact match and a SAGE system at 1.77%; `M077` and `M078` record DS-IA at 58.56% exact match with a 74.90 F1 score. `M083` then records the finding that matters most for reading those numbers: a constant 'never act' predictor already reaches 82% accuracy. The reference line below is that critic, drawn on the same axis as the results it undermines.

In [ ]:
# M079 states two values in one cell ("29.98 and 1.77"), so parse_numbers keeps
# them in the order the cell writes them and the labels say which is which.
# Both DS-IA figures are drawn because exact match and F1 are different
# measures of the same system, not two systems.
_results = viz.metrics_by_id(["M079", "M077", "M078", "M083"])
_by_id = _results.set_index("local_id")
_ms = viz.parse_numbers(_by_id.loc["M079", "value"])
_systems = pd.DataFrame([
    {"system": "Baseline (M079, value 1 of 2)", "pct": _ms[0] if _ms else None,
     "measure": "exact match", "source": _by_id.loc["M079", "source_ref"]},
    {"system": "SAGE (M079, value 2 of 2)",
     "pct": _ms[1] if len(_ms) > 1 else None,
     "measure": "exact match", "source": _by_id.loc["M079", "source_ref"]},
    {"system": "DS-IA (M077)", "pct": viz.numeric_value(_by_id.loc["M077", "value"]),
     "measure": "exact match", "source": _by_id.loc["M077", "source_ref"]},
    {"system": "DS-IA (M078)", "pct": viz.numeric_value(_by_id.loc["M078", "value"]),
     "measure": "F1 score", "source": _by_id.loc["M078", "source_ref"]},
])
show(_systems)

_never_act = viz.numeric_value(_by_id.loc["M083", "value"])
fig = go.Figure(go.Bar(
    x=_systems["system"],
    y=_systems["pct"],
    marker_color=["#757575", "#757575", "#1f77b4", "#7aa6d2"],
    text=["%g%%" % value for value in _systems["pct"]],
    textposition="outside",
    customdata=list(zip(_systems["measure"], _systems["source"])),
    hovertemplate=("%{x}<br>%{customdata[0]}: %{y}%"
                   "<br>source: %{customdata[1]}<extra></extra>"),
))
if _never_act is not None:
    fig.add_hline(
        y=_never_act,
        line_dash="dot",
        line_color="#b06000",
        annotation_text=(f"M083: constant 'never act' predictor already "
                         f"reaches {_never_act:g}%"),
        annotation_position="bottom right",
    )
fig.update_layout(
    title=("Reported accuracy on home-AI intent benchmarks, with the "
           "do-nothing baseline drawn (M077-M079, M083)"),
    yaxis_title="percent as stated (exact match or F1, per bar)",
    height=380,
    xaxis=dict(tickangle=-15),
)
fig.show()

# Counts of what the benchmarks contain. These bars are deliberately not read
# as a comparison: they count different objects recorded by different artefacts.
DESIGN_IDS = ["M071", "M072", "M073", "M084", "M085"]
design = viz.metrics_by_id(DESIGN_IDS)
show(design[["local_id", "metric_name", "value", "unit", "confidence",
             "source_ref", "section"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "source_ref": "source", "confidence": "conf"}))
fig = go.Figure(go.Bar(
    x=design["local_id"] + " " + design["metric_name"],
    y=[viz.numeric_value(value) for value in design["value"]],
    marker_color=[viz.CONFIDENCE_COLORS.get(c, viz.CONFIDENCE_COLORS[None])
                  for c in design["confidence"]],
    text=[viz.numeric_value(value) for value in design["value"]],
    textposition="outside",
    customdata=list(zip(design["value"], design["unit"], design["source_ref"])),
    hovertemplate=("%{x}<br>stated: %{customdata[0]} %{customdata[1]}"
                   "<br>source: %{customdata[2]}<extra></extra>"),
))
fig.update_layout(
    title=("Scale of the recorded test designs: five counts of five different "
           "objects (not a comparison)"),
    yaxis_title="count of the unit named in the hover",
    height=380,
    xaxis=dict(tickangle=-20),
)
fig.show()

## Supporting context

### Safety-relevant measures that are not accuracies

Four recorded quantities in this cluster are not success rates: the Cascade verifier's rejection rate for invalid instructions (`M080`), a dependency-satisfaction ratio (`M081`), an unsafe-execution rate (`M082`) and a direct-command accuracy for local models (`M076`). They are kept as a table with their own units because a ratio, a rate and a percentage cannot share an axis without implying a common scale. `M074` and `M075` record the benchmark rubric's weights, which is why the cluster's scoring is not a single number.

In [ ]:
display(HTML(
    "<h4>Safety measures recorded in units other than percent</h4>"
    "<p>Each row is one metric; the unit column is the corpus's own. A ratio "
    "and a rate are not percentages, so they are not drawn on the percent "
    "axis above.</p>"
))
_measures = viz.metrics_by_id(["M080", "M081", "M082", "M076", "M074", "M075"])
show(_measures[["local_id", "metric_name", "value", "unit", "confidence",
                "source_ref", "section"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "source_ref": "source", "confidence": "conf"}))

display(HTML("""
<h4>What the cluster does not contain</h4>
<p><code>C041</code> states that no unified harness evaluates all fourteen
scenario categories across systems with a consistent rubric, and
<code>C042</code> states that commercial assistants publish no benchmark
results at all. The corpus therefore records research artefacts and
methodology claims (<code>C106</code>, <code>C107</code>, <code>C108</code>),
not comparative product results.</p>
"""))
display(HTML(
    "<h4>Claim cluster evidence table</h4>"
    "<p>Every claim in this cluster, with its evidence class, confidence, "
    "workflow stage and cited source ids. This is the table the figures above "
    "are built from.</p>"
))
show(
    claims[["local_id", "claim_type", "evidence_marker", "confidence",
            "workflow_stage", "source_refs", "falsifier_missing", "section"]]
    .rename(columns={
        "local_id": "claim", "claim_type": "type", "evidence_marker": "ev",
        "confidence": "conf", "workflow_stage": "stage",
        "source_refs": "sources", "falsifier_missing": "no_falsifier",
    })
)

display(HTML("<h4>Sources cited by the cluster</h4>"))
show(sources[["local_id", "title", "publisher", "classification",
              "publication_date", "url"]].rename(
    columns={"local_id": "source"}))

## Uncertainty and gaps

### The numbers above come from five different artefacts

DS-IA (`S47`), PromptShield Home (`S48`), the dependency-aware orchestration work (`S49`), SimVerity (`S52`) and the Evil-AI adversarial corpus (`S78`) are separate research efforts with separate test sets. The bars on the first chart compare systems *within* one artefact's evaluation; the design-count chart spans all of them and is explicitly not a comparison. The corpus records no common test set, which is exactly what `C041` complains about.

Other limits on this page:

- **A high accuracy can be a red flag.** `M083` records that a constant 'never act' predictor reaches 82% accuracy, which is why unsafe-execution rates (`M082`, `0.02`) matter more than aggregate accuracy when the task is safety-critical.
- **`C031`'s failure modes are low confidence.** Hallucinated device state, mistaken identity and over-broad automation are recorded as reported signal at low confidence, so they are described rather than charted.
- **`C119` records a framework with no implementation.** The 'Circles of Trust' privilege model is proposed in research; the corpus records no commercial product that ships it.
- **Nothing here measures a shipping assistant.** `C042` is explicit that Alexa, Google Home, Siri and SmartThings publish no safety benchmark results, so no figure on this page can be used to rank them.

In [ ]:
# Benchmark-existence and methodology claims are separated from the results,
# so a claim that a benchmark exists is never read as a result from it.
display(HTML(
    "<h4>Methodology claims in this cluster, with their evidence class</h4>"
    "<p>A claim that a benchmark exists is not a claim about what it "
    "found.</p>"
))
show(claims.loc[claims["local_id"].isin(
        ["C041", "C042", "C106", "C107", "C108", "C119"]),
    ["local_id", "claim_type", "confidence", "workflow_stage",
     "source_refs", "falsifier_missing", "claim_text"]].rename(
    columns={"local_id": "claim", "claim_type": "type", "confidence": "conf",
             "workflow_stage": "stage", "source_refs": "sources",
             "falsifier_missing": "no_falsifier"}))
display(HTML("<h4>Claims in this cluster with no falsifier</h4>"
             "<p>A missing falsifier means the corpus recorded no observation "
             "that would disconfirm the claim. The panel above flags these "
             "claims in <code>no_falsifier</code>.</p>"))
show(claims.loc[claims["falsifier_missing"], ["local_id", "claim_type",
                                              "confidence"]]
     .rename(columns={"local_id": "claim", "claim_type": "type",
                      "confidence": "conf"}))

display(HTML("<h4>Metrics with conditions not stated in the source</h4>"))
_gaps = cluster_metrics.loc[~cluster_metrics["conditions_stated"].fillna(False)]
if _gaps.empty:
    display(HTML("<p>No unstated conditions among the cluster's metrics.</p>"))
else:
    show(_gaps[["local_id", "metric_name", "value", "unit"]].rename(
        columns={"local_id": "metric", "metric_name": "name"}))

display(HTML("<h4>Extraction decisions that changed or qualified this material</h4>"))
_decisions = pd.DataFrame(
    viz.connect_db().execute(
        "SELECT local_id, decision_type, description, resolution "
        "FROM extraction_decision "
        "WHERE decision_type IN ('evidence_downgrade', 'classification_conflict', "
        "                           'falsifier_absent', 'condition_absent', "
        "                           'boundary_absent', 'scope_boundary', "
        "                           'omission') "
        "ORDER BY local_id"
    ).fetchall(),
    columns=["decision", "type", "description", "resolution"],
)
show(_decisions)

## Reproducibility

Every number is read by explicit metric id with `viz.metrics_by_id` and parsed with `viz.parse_numbers` / `viz.numeric_value`, because the value cells are authored strings ('29.98 and 1.77', 'up to 86.7'). Claim text and evidence classes come from `viz.fetch_claims`. No result is transcribed from the markdown corpus by hand.

In [ ]:
# The exact queries behind every figure above, re-run verbatim.
import duckdb
con = duckdb.connect(str(ROOT / 'data' / 'smarthome.duckdb'), read_only=True)
print('database:', con.execute('SELECT current_database()').fetchone()[0])
for table in ('stage2_claim', 'metric', 'entity', 'source_mirror',
               'predicate', 'workflow_stage', 'extraction_decision'):
    total = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:22} {total:>6} rows')
con.close()